# Thống kê bằng code với ASAO ERP

Notebook này đọc dữ liệu Odoo qua CLI của ASAO (`asao_erp`). Notebook không giữ mật khẩu: profile và mật khẩu
nằm ở cấu hình của CLI (biến môi trường hoặc keyring). Client chỉ đọc; muốn ghi phải tạo client với
`allow_writes=True`.

Trước khi commit: **xoá output** (Clear All Outputs), vì output chứa dữ liệu khách hàng.

In [ ]:
import os
from asao_erp import Client, ops, to_frame

PROFILE = os.environ.get("ASAO_ERP_PROFILE", "asao-hq")
erp = Client(profile=PROFILE, allow_unverified=True)  # chỉ đọc

## Doanh thu theo tháng (đơn đã xác nhận)

Odoo tự gộp nhóm và tính tổng; notebook chỉ nhận kết quả.

In [ ]:
doanh_thu = to_frame(erp.group(
    ops.sales.quotation_list,
    by=["date_order:month"],
    aggregate=["amount_untaxed:sum", "amount_total:sum"],
    filter=[["state", "=", "sale"]],
))
doanh_thu

## Công nợ phải thu theo khách hàng (10 khách nhiều nhất)

In [ ]:
cong_no = to_frame(erp.group(
    ops.finance.invoice_list,
    by=["partner_id"],
    aggregate=["amount_residual:sum"],
    filter=[["move_type", "=", "out_invoice"], ["state", "=", "posted"], ["payment_state", "in", ["not_paid", "partial"]]],
    order="amount_residual:sum desc",
    limit=10,
))
cong_no

## Pipeline theo giai đoạn

In [ ]:
pipeline = to_frame(erp.group(
    ops.crm.lead_list,
    by=["stage_id"],
    aggregate=["expected_revenue:sum"],
    filter=[["type", "=", "opportunity"]],
))
pipeline

## Toàn bộ hoá đơn bán, ra bảng và CSV

`iterate` đọc lần lượt từng trang cho tới hết.

In [ ]:
hoa_don = to_frame(erp.iterate(ops.finance.invoice_list, [["move_type", "=", "out_invoice"]]))
hoa_don.to_csv("hoa-don.csv", index=False)
hoa_don[["name", "partner", "invoice_date", "total_amount", "payment_status"]].head()

## In một hoá đơn ra PDF

Odoo chỉ trả PDF cho phiên đăng nhập web, nên CLI cần mật khẩu web của profile:
`AERP_CREDENTIAL_<PROFILE>_WEB` (hoặc keyring `<profile>.web`).

In [ ]:
mot_hoa_don = int(hoa_don["id"].iloc[0])
erp.print(ops.finance.invoice_print, mot_hoa_don, to=f"hoa-don-{mot_hoa_don}.pdf")